In [19]:
import httpx

GO_ACIS = "http://localhost:8080"
REAL_ACIS = "https://data.rcc-acis.org/StnData"
client = httpx.Client(verify=False)

#### Test health endpoint

In [10]:
r = client.get(GO_ACIS + "/health")
print(r.json())

{'status': 'healthy'}


#### Test StnData endpoint 

struct def for reference:

```
type StnDataRequest struct {
	SID      string      `json:"sid"`
	SDate    string      `json:"sdate,omitempty"`
	EDate    string      `json:"edate,omitempty"`
	Date     string      `json:"date,omitempty"`
	Elements []Element   `json:"elems"`
	Meta     *[]string   `json:"meta,omitempty"`
	Output   *OutputType `json:"output,omitempty"`
}
```

In [18]:
stn_payload = {
    "sid" : "KRDU", # Raleigh
    "sdate" : "2017-04-17",
    "edate" : "2018-04-17",
    "elems" : [{"name" : "maxt"}]
}

r = client.post(GO_ACIS + "/api/stn-data", json=stn_payload)
print(r.status_code)
print(r.json())

502
{'error': 'encoding error: json: cannot unmarshal array into Go struct field StnDataResponse.data of type acis.DataRow'}


In [24]:
stn_payload = {
    "sid" : "KRDU", # Raleigh
    "sdate" : "2017-04-17",
    "edate" : "2018-04-17",
    "elems" : [{"name" : "maxt"}]
}

r = client.post(REAL_ACIS, json=stn_payload)
print(r.status_code)
print(r.json()["data"])

200
[['2017-04-17', '86'], ['2017-04-18', '74'], ['2017-04-19', '69'], ['2017-04-20', '85'], ['2017-04-21', '89'], ['2017-04-22', '87'], ['2017-04-23', '56'], ['2017-04-24', '60'], ['2017-04-25', '70'], ['2017-04-26', '81'], ['2017-04-27', '82'], ['2017-04-28', '88'], ['2017-04-29', '89'], ['2017-04-30', '86'], ['2017-05-01', '85'], ['2017-05-02', '77'], ['2017-05-03', '79'], ['2017-05-04', '80'], ['2017-05-05', '78'], ['2017-05-06', '67'], ['2017-05-07', '69'], ['2017-05-08', '71'], ['2017-05-09', '64'], ['2017-05-10', '83'], ['2017-05-11', '84'], ['2017-05-12', '61'], ['2017-05-13', '66'], ['2017-05-14', '81'], ['2017-05-15', '85'], ['2017-05-16', '87'], ['2017-05-17', '89'], ['2017-05-18', '86'], ['2017-05-19', '89'], ['2017-05-20', '90'], ['2017-05-21', '76'], ['2017-05-22', '80'], ['2017-05-23', '71'], ['2017-05-24', '74'], ['2017-05-25', '76'], ['2017-05-26', '81'], ['2017-05-27', '88'], ['2017-05-28', '86'], ['2017-05-29', '87'], ['2017-05-30', '82'], ['2017-05-31', '86'], ['201

Current DataRow definition in response.go does not match what ACIS actually returns. Our response object expects data to come back in an object like


`{
    "Date": "2017-04-17",
    "Values": [...]
  }
`


but ACIS returns each data row as an array where the 0 index is the date, and consequent indices represent the values. this is shown in the cell above. We need to either implement a custom decoder to force into the DataRow shape you designed, or update the StnDataResponseBody to reflect this. 